In [5]:
from google.colab import files
import os

# Create directories
input_folder = 'invoices'
output_folder = 'translated_invoices'

if not os.path.exists(input_folder):
    os.makedirs(input_folder)

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Upload files
uploaded = files.upload()
for file_name in uploaded.keys():
    file_path = os.path.join(input_folder, file_name)
    with open(file_path, 'wb') as f:
        f.write(uploaded[file_name])


Saving CGAP-Glossary-English-to-Chinese-Jul-2008_0-1 (2).pdf to CGAP-Glossary-English-to-Chinese-Jul-2008_0-1 (2).pdf


In [2]:
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 49 not upgraded.
Need to get 186 kB of archives.
After this operation, 696 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.5 [186 kB]
Fetched 186 kB in 1s (317 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 123597 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.5_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.5) ...
Setting up poppler-utils (22.02.0-2ubuntu0.5) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
# Install the necessary packages
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y libtesseract-dev
!pip install pytesseract
!pip install pdf2image
!pip install langdetect
!pip install googletrans==4.0.0-rc1
!pip install pillow


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Ign:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [962 kB]
Get:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ge

In [6]:
import pytesseract
from pdf2image import convert_from_path
from langdetect import detect
from googletrans import Translator
from PIL import Image
import os

# Setup paths for OCR
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'  # Update with the correct path to Tesseract OCR on your machine

# Initialize the translator
translator = Translator()

def extract_text_from_image(image_path):
    image = Image.open(image_path)
    return pytesseract.image_to_string(image)

def extract_text_from_pdf(pdf_path):
    images = convert_from_path(pdf_path)
    text = ""
    for image in images:
        text += pytesseract.image_to_string(image)
    return text

def detect_and_translate(text, target_language="en"):
    try:
        detected_language = detect(text)
        if detected_language != target_language:
            translated_text = translator.translate(text, src=detected_language, dest=target_language).text
            return translated_text
        else:
            return text
    except Exception as e:
        print(f"Error detecting or translating text: {e}")
        return text

def process_invoice(file_path):
    if file_path.endswith(".pdf"):
        text = extract_text_from_pdf(file_path)
    elif file_path.endswith((".png", ".jpg", ".jpeg")):
        text = extract_text_from_image(file_path)
    else:
        raise ValueError("Unsupported file format")

    translated_text = detect_and_translate(text)
    return translated_text

# Example usage
input_folder = 'invoices'  # Path to your invoices
output_folder = 'translated_invoices'  # Path to save translated invoices

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for file_name in os.listdir(input_folder):
    file_path = os.path.join(input_folder, file_name)
    try:
        translated_text = process_invoice(file_path)
        output_file_path = os.path.join(output_folder, file_name.replace('.pdf', '.txt').replace('.jpg', '.txt').replace('.png', '.txt'))
        with open(output_file_path, 'w') as f:
            f.write(translated_text)
        print(f"Processed and translated {file_name}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")


Processed and translated Proforma_Invoice_French.pdf
Processed and translated CGAP-Glossary-English-to-Chinese-Jul-2008_0-1 (2).pdf
